# Rishu Kumar (Spam classification)

### Importing the dataset

In [ ]:
import pandas as pd

message = pd.read_csv('SMSSpamCollection', sep = '\t', names = ["label", "message"])
# 1st col specifies whether the message is ham or spam
# i am giving name of the columns i.e label and message


In [ ]:
message.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
message.shape

(5572, 2)

In [ ]:
len(message)

5572

In [ ]:
print(message)

     label                                            message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham               Will ü b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


### Data cleaning and preprocessing

In [ ]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(len(message)):
  review = re.sub('[^a-zA-Z0-9]',' ', message["message"][i])
  # print("After removing regular expression:", review)
  review = review.lower()
  # print("After converting to lowercase:", review)
  review = review.split()

  review = [ps.stem(word) for word in review if not word in set(stopwords.words('english'))]
  review = ' '.join(review)
  corpus.append(review)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Creating the Bag of Words model

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# converting text data into numerical format, we can use this word2vec also
# Converts a collection of text documents into a matrix of token (word) counts.
#It basically creates a Bag of Words (BoW) model.

cv = CountVectorizer(max_features = 3000)
x = cv.fit_transform(corpus).toarray()
print(x[1])

y = pd.get_dummies(message["label"])
# converts categorical labels into one-hot encoded vectors
print(y)
y = y.iloc[:,1].values
print(y)

[0 0 0 ... 0 0 0]
        ham   spam
0      True  False
1      True  False
2     False   True
3      True  False
4      True  False
...     ...    ...
5567  False   True
5568   True  False
5569   True  False
5570   True  False
5571   True  False

[5572 rows x 2 columns]
[False False  True ... False False False]


### Train Test and Split

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state = 0)

### Training model using Naive bayes classifier

In [ ]:
# naive bayes is a classification technique and it uses probability to classify the different classes.

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

models = {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "RandomForestClassifier": RandomForestClassifier()
}

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    print(f"\n{name} Predictions: {y_pred}")

    # printing accuracy and confusion matrix for each model
    print(f"{name} Results:")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Accuracy Score:", accuracy_score(y_test, y_pred))


MultinomialNB Predictions: [False  True False ... False  True False]
MultinomialNB Results:
Confusion Matrix:
 [[948   7]
 [  8 152]]
Accuracy Score: 0.9865470852017937

LogisticRegression Predictions: [False  True False ... False  True False]
LogisticRegression Results:
Confusion Matrix:
 [[954   1]
 [ 19 141]]
Accuracy Score: 0.9820627802690582

DecisionTreeClassifier Predictions: [False  True False ... False  True False]
DecisionTreeClassifier Results:
Confusion Matrix:
 [[941  14]
 [ 20 140]]
Accuracy Score: 0.9695067264573991

RandomForestClassifier Predictions: [False  True False ... False  True False]
RandomForestClassifier Results:
Confusion Matrix:
 [[954   1]
 [ 16 144]]
Accuracy Score: 0.9847533632286996


### Sample Data

In [ ]:
sample_message = ["Congratulations! You have won a free lottery. Call now to claim."]
sample_message_processed = [re.sub('[^a-zA-Z0-9]', ' ', sample_message[0]).lower()]
sample_message_vectorized = cv.transform(sample_message_processed).toarray()

# MultinomialNB performs better here in this case
best_model = MultinomialNB()
best_model.fit(x_train, y_train)
sample_prediction = best_model.predict(sample_message_vectorized)
print(sample_prediction[0])

print("Sample Message:", sample_message[0])
print("Prediction:", "Spam" if sample_prediction == 1 else "Ham")

True
Sample Message: Congratulations! You have won a free lottery. Call now to claim.
Prediction: Spam
